In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Persistence baseline for NO2
# Idea: for each pixel-day, predict today's NO2 using yesterday's NO2.
# This is a strong sanity-check baseline before trying any neural network.
# check baseline before trying any neural network.

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1) Choose the target and the persistence feature.
# In this lagged table:
# - no2_day0 = today's NO2 (the thing we want to predict)
# - no2_day1 = yesterday's NO2 (the persistence forecast)
target_col = "no2_day0"
pred_col = "no2_day1"

# 2) Keep only rows where both target and persistence input exist.
baseline_df = no2_lagged_df[["timestamp", "latitude", "longitude", target_col, pred_col]].dropna().copy()
baseline_df = baseline_df.sort_values("timestamp").reset_index(drop=True)

# 3) Make a time-based split so we evaluate on future dates, not shuffled rows.
unique_dates = baseline_df["timestamp"].drop_duplicates().sort_values().to_list()
cutoff_index = max(1, int(len(unique_dates) * 0.8))
train_dates = set(unique_dates[:cutoff_index])
test_dates = set(unique_dates[cutoff_index:])

train_df = baseline_df[baseline_df["timestamp"].isin(train_dates)].copy()
test_df = baseline_df[baseline_df["timestamp"].isin(test_dates)].copy()

# 4) Evaluate the baseline on the held-out future period.
y_true = test_df[target_col].to_numpy()
y_pred = test_df[pred_col].to_numpy()

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

# A simple normalized error helps when NO2 values vary a lot across pixels.
mean_abs_target = np.mean(np.abs(y_true))
nmae = mae / mean_abs_target if mean_abs_target != 0 else np.nan

print("Persistence baseline for NO2")
print(f"Train rows: {len(train_df):,}")
print(f"Test rows:  {len(test_df):,}")
print(f"Time split:  {min(test_df['timestamp']).date()} to {max(test_df['timestamp']).date()}")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2:  {r2:.4f}")
print(f"NMAE: {nmae:.4f}")

# # 5) Save predictions so you can inspect where persistence works well or fails.
# baseline_predictions = test_df[["timestamp", "latitude", "longitude", target_col, pred_col]].rename(
#     columns={target_col: "no2_true", pred_col: "no2_persistence_pred"}
# )

# 6) Quick diagnostic plot: actual vs predicted.
plt.figure(figsize=(6, 6))
plt.scatter(y_true, y_pred, alpha=0.15, s=10)
min_val = min(np.min(y_true), np.min(y_pred))
max_val = max(np.max(y_true), np.max(y_pred))
plt.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=2)
plt.xlabel("Actual NO2")
plt.ylabel("Persistence prediction")
plt.title("Persistence baseline: actual vs predicted NO2")
plt.grid(True, alpha=0.3)
plt.show()

# baseline_predictions.head()